# 4. Feature Engineering — Расширенная генерация признаков

## Цель ноутбука

В этом ноутбуке мы проведем **полноценный Feature Engineering** для задачи предсказания цен на автомобили Craigslist. На основе разведочного анализа данных (EDA) из ноутбука `02_eda.ipynb` мы:

1. Создадим новые признаки (возраст автомобиля, логарифмические преобразования)
2. Обработаем категориальные признаки (One-Hot Encoding, Target Encoding)
3. Сгруппируем редкие категории
4. Создадим признаки по региону и производителю
5. Добавим полиномиальные и интерактивные признаки
6. Применим скейлинг числовых признаков
7. Проведем отбор признаков

---

## Содержание

1. [Загрузка данных и библиотек](#section-1)
2. [Базовые преобразования](#section-2)
   - Возраст машины
   - Логарифм цены и одометра
3. [Обработка категориальных признаков](#section-3)
   - Заполнение пропусков
   - Группировка редких категорий
   - One-Hot Encoding
   - Target Encoding
4. [Признаки по региону и производителю](#section-4)
   - Ценовые кластеры регионов
   - Статистики по производителям
5. [Полиномиальные и интерактивные признаки](#section-5)
6. [Скейлинг числовых признаков](#section-6)
7. [Отбор признаков](#section-7)
8. [Сохранение результатов](#section-8)

<a id='section-1'></a>
## 1. Загрузка данных и библиотек

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.cluster import KMeans

# Настройки визуализации
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)

print("Библиотеки загружены")

In [ ]:
data_path = Path("../data/interim/train_filtered.csv")
df = pd.read_csv(data_path, index_col=0)

print(f"Размер датасета: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
df.head(3)

In [ ]:
X = df.drop(columns=['price']).copy()
y = df['price'].copy()

y_log = np.log1p(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Статистика цены:")
print(y.describe())

<a id='section-2'></a>
## 2. Базовые преобразования

### 2.1 Возраст машины (`car_age`)

**Обоснование:** Из EDA видно, что год выпуска сильно коррелирует с ценой. Однако абсолютный год менее информативен, чем **возраст автомобиля** — разница между текущим годом и годом выпуска.

**Формула:** `car_age = current_year - year`

In [ ]:
CURRENT_YEAR = 2022  # Год, относительно которого считаем возраст

X['year'] = X['year'].fillna(X['year'].mode()[0])

X['car_age'] = CURRENT_YEAR - X['year']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(X['year'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Распределение года выпуска', fontsize=14)
axes[0].set_xlabel('Год')

sns.histplot(X['car_age'], bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Распределение возраста автомобилей', fontsize=14)
axes[1].set_xlabel('Возраст (лет)')

plt.tight_layout()
plt.show()

print(f"Статистика возраста:\n{X['car_age'].describe()}")

### 2.2 Логарифмическое преобразование числовых признаков

**Обоснование:** Из EDA видно, что распределения `price` и `odometer` имеют сильную правостороннюю асимметрию (long right tail). Логарифмическое преобразование:
- Уменьшает асимметрию
- Делает распределение более нормальным
- Уменьшает влияние выбросов
- Улучшает работу многих моделей

**Преобразования:**
- `log_price = log(1 + price)` — целевая переменная
- `log_odometer = log(1 + odometer)` — признак пробега

In [ ]:
X['log_odometer'] = np.log1p(X['odometer'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(X['odometer'], bins=100, kde=True, ax=axes[0], color='green')
axes[0].set_title('Распределение одометра (до)', fontsize=14)
axes[0].set_xlabel('Пробег (мили)')

sns.histplot(X['log_odometer'], bins=100, kde=True, ax=axes[1], color='purple')
axes[1].set_title('Распределение log(1 + odometer) (после)', fontsize=14)
axes[1].set_xlabel('log(Пробег)')

plt.tight_layout()
plt.show()

from scipy.stats import skew

skew_before = skew(X['odometer'].dropna())
skew_after = skew(X['log_odometer'].dropna())

print(f"Асимметрия одометра: {skew_before:.2f} -> {skew_after:.2f}")
print(f"Статистика log_odometer:\n{X['log_odometer'].describe()}")

In [ ]:
y_log = np.log1p(y)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(y, bins=100, kde=True, ax=axes[0], color='orange')
axes[0].set_title('Распределение цены (до)', fontsize=14)
axes[0].set_xlabel('Цена ($)')

sns.histplot(y_log, bins=100, kde=True, ax=axes[1], color='red')
axes[1].set_title('Распределение log(1 + price) (после)', fontsize=14)
axes[1].set_xlabel('log(Цена)')

plt.tight_layout()
plt.show()

print(f"Асимметрия цены: {skew(y):.2f} -> {skew(y_log):.2f}")

### 2.3 Преобразование цилиндров в числовой признак

In [ ]:
CYLINDERS_MAP = {
    "3 cylinders": 3,
    "4 cylinders": 4,
    "5 cylinders": 5,
    "6 cylinders": 6,
    "8 cylinders": 8,
    "10 cylinders": 10,
    "12 cylinders": 12,
    "other": 0
}

X['cylinders'] = X['cylinders'].fillna('other')
X['cylinders_num'] = X['cylinders'].map(CYLINDERS_MAP).fillna(0).astype(int)

X['cylinders_other_flag'] = (X['cylinders'] == 'other').astype(int)

print(f"Распределение цилиндров:\n{X['cylinders_num'].value_counts().sort_index()}")
print(f"Флаг other: {X['cylinders_other_flag'].sum()} записей")

<a id='section-3'></a>
## 3. Обработка категориальных признаков

### 3.1 Заполнение пропусков

In [ ]:
missing_before = X.isnull().sum()
missing_pct_before = (X.isnull().mean() * 100).round(2)

missing_df_before = pd.DataFrame({
    'Missing Count': missing_before,
    'Missing %': missing_pct_before
}).sort_values('Missing %', ascending=False)

print("Пропуски ДО обработки:")
print(missing_df_before[missing_df_before['Missing Count'] > 0])

In [ ]:
CATEGORICAL_COLS_TO_FILL = [
    'condition', 'drive', 'transmission', 'fuel', 
    'manufacturer', 'title_status', 'type', 'model'
]

for col in CATEGORICAL_COLS_TO_FILL:
    if col in X.columns:
        X[col] = X[col].fillna('unknown')

missing_after = X.isnull().sum()
print("Пропуски заполнены!")
print("Пропуски ПОСЛЕ обработки:")
print(missing_after[missing_after > 0])

### 3.2 Группировка редких категорий

**Обоснование:** Многие модели плохо работают с признаками, имеющими большое количество редких категорий. Это приводит к:
- Переобучению на редких категориях
- Увеличению размерности при One-Hot Encoding
- Плохой обобщающей способности

**Стратегия:** Категории, встречающиеся реже порога, объединяем в `'other'`

In [ ]:
def group_rare_categories(df, column, min_count=20):
    """
    Группирует редкие категории в 'other'.
    
    Parameters:
    -----------
    df : DataFrame
    column : str — имя колонки
    min_count : int — минимальное количество вхождений для сохранения категории
    
    Returns:
    --------
    Series — преобразованная колонка
    """
    value_counts = df[column].value_counts()
    rare_categories = value_counts[value_counts < min_count].index
    
    result = df[column].copy()
    result[result.isin(rare_categories)] = 'other'
    
    return result, len(rare_categories)

X['model_grouped'], n_rare_models = group_rare_categories(X, 'model', min_count=20)

print(f"Model: {X['model'].nunique()} уникальных значений -> {X['model_grouped'].nunique()} после группировки")
print(f"   Сгруппировано редких категорий: {n_rare_models}")
print(f"Топ-15 моделей после группировки:")
print(X['model_grouped'].value_counts().head(15))

In [ ]:
if X['manufacturer'].nunique() > 50:
    X['manufacturer_grouped'], n_rare_manuf = group_rare_categories(X, 'manufacturer', min_count=100)
    print(f"Manufacturer: {X['manufacturer'].nunique()} -> {X['manufacturer_grouped'].nunique()}")
    print(f"   Сгруппировано редких: {n_rare_manuf}")
else:
    X['manufacturer_grouped'] = X['manufacturer']
    print("Manufacturer не требует группировки")

### 3.3 One-Hot Encoding для категориальных признаков

**Обоснование:** Большинство моделей машинного обучения требуют числовые признаки. One-Hot Encoding создает бинарные признаки для каждой категории.

**Важно:** Применяем только к признакам с небольшим количеством уникальных значений (< 10-15), чтобы избежать проклятия размерности.

In [ ]:
LOW_CARDINALITY_COLS = [
    'condition',      # ~6 категорий
    'transmission',   # ~3 категории
    'drive',          # ~4 категории
    'fuel',           # ~5 категорий
    'title_status',   # ~4 категории
    'type',           # ~9 категорий
]

print("Кардинальность категориальных признаков:")
for col in LOW_CARDINALITY_COLS:
    n_unique = X[col].nunique()
    print(f"   {col}: {n_unique} уникальных значений")

In [ ]:
X_ohe = pd.get_dummies(
    X, 
    columns=LOW_CARDINALITY_COLS,
    prefix=LOW_CARDINALITY_COLS,
    drop_first=False,  # Оставляем все категории для интерпретируемости
    dtype=int
)

print(f"Размерность до OHE: {X.shape}")
print(f"Размерность после OHE: {X_ohe.shape}")
print(f"   Добавлено признаков: {X_ohe.shape[1] - X.shape[1]}")

### 3.4 Target Encoding для признаков с высокой кардинальностью

**Обоснование:** Для признаков с большим количеством категорий (model, manufacturer, region) One-Hot Encoding создаст слишком много признаков. **Target Encoding** заменяет каждую категорию средним значением целевой переменной для этой категории.

**Преимущества:**
- Сохраняет информацию о связи с целевой переменной
- Не увеличивает размерность
- Работает с высокой кардинальностью

**Важно:** Чтобы избежать утечки данных, используем сглаживание (smoothing) и вычисляем encoding только на тренировочных данных.

In [ ]:
def target_encode(train_df, test_df, column, target, smoothing=10):
    """
    Target Encoding со сглаживанием.
    
    Formula: 
    encoded_value = (n * category_mean + smoothing * global_mean) / (n + smoothing)
    
    где n — количество наблюдений в категории
    """
    global_mean = train_df[target].mean()
    
    agg = train_df.groupby(column)[target].agg(['mean', 'count'])
    
    agg['smoothed_mean'] = (
        agg['mean'] * agg['count'] + smoothing * global_mean
    ) / (agg['count'] + smoothing)
    
    mapping = agg['smoothed_mean'].to_dict()
    
    train_encoded = train_df[column].map(mapping).fillna(global_mean)
    test_encoded = test_df[column].map(mapping).fillna(global_mean)
    
    return train_encoded, test_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X_ohe, y_log, test_size=0.2, random_state=42
)

HIGH_CARDINALITY_COLS = ['manufacturer_grouped', 'model_grouped', 'region']

for col in HIGH_CARDINALITY_COLS:
    if col in X_train.columns:
        train_enc, test_enc = target_encode(X_train, X_test, col, 'price', smoothing=20)
        X_train[f'{col}_target'] = train_enc
        X_test[f'{col}_target'] = test_enc

X_te = pd.concat([X_train, X_test]).sort_index()

print(f"Target Encoding применен к: {HIGH_CARDINALITY_COLS}")
print(f"Новые признаки: {[f'{col}_target' for col in HIGH_CARDINALITY_COLS]}")

<a id='section-4'></a>
## 4. Признаки по региону и производителю

### 4.1 Кластеризация регионов по ценам

**Обоснование:** Регионы могут иметь разные ценовые уровни. Кластеризация поможет выявить группы регионов со схожими ценами.

In [ ]:
region_price = df.groupby('region')['price'].agg(['mean', 'median', 'std', 'count']).reset_index()
region_price.columns = ['region', 'region_mean_price', 'region_median_price', 'region_price_std', 'region_count']

X_te = X_te.merge(region_price[['region', 'region_mean_price', 'region_median_price', 'region_price_std']], 
                  on='region', how='left')

print(f"Количество регионов: {region_price.shape[0]}")
print(f"Топ-10 дорогих регионов:")
print(region_price.nlargest(10, 'region_mean_price')[['region', 'region_mean_price']])

In [ ]:
N_REGION_CLUSTERS = 5

kmeans = KMeans(n_clusters=N_REGION_CLUSTERS, random_state=42)
region_price['region_cluster'] = kmeans.fit_predict(region_price[['region_mean_price', 'region_price_std']].fillna(0))

X_te = X_te.merge(region_price[['region', 'region_cluster']], on='region', how='left')

print(f"Регионы сгруппированы в {N_REGION_CLUSTERS} кластеров")
print(f"Распределение по кластерам:")
print(X_te['region_cluster'].value_counts().sort_index())

### 4.2 Статистики по производителю

**Обоснование:** Разные производители имеют разные ценовые позиционирования. Создадим признаки, отражающие среднюю цену и разброс цен для каждого производителя.

In [ ]:
manuf_stats = df.groupby('manufacturer')['price'].agg(['mean', 'median', 'std', 'count']).reset_index()
manuf_stats.columns = ['manufacturer', 'manuf_mean_price', 'manuf_median_price', 'manuf_price_std', 'manuf_count']

X_te = X_te.merge(manuf_stats[['manufacturer', 'manuf_mean_price', 'manuf_median_price', 'manuf_price_std']], 
                  on='manufacturer', how='left')

print(f"Созданы признаки по производителю:")
print("   - manuf_mean_price: средняя цена производителя")
print("   - manuf_median_price: медианная цена производителя")
print("   - manuf_price_std: стандартное отклонение цен производителя")

### 4.3 Обработка координат

**Обоснование:** Координаты (lat, long) могут быть полезны, но в сыром виде модели их плохо используют. Создадим признаки расстояния до центра.

In [ ]:
region_coords = df.groupby('region')[['lat', 'long']].mean().reset_index()

X_te = X_te.merge(region_coords, on='region', how='left', suffixes=('', '_region_mean'))

X_te['dist_to_region_center'] = np.sqrt(
    (X_te['lat'] - X_te['lat_region_mean'])**2 + 
    (X_te['long'] - X_te['long_region_mean'])**2
)

print(f"Координаты обработаны")
print(f"Среднее расстояние до центра региона: {X_te['dist_to_region_center'].mean():.4f}")

<a id='section-5'></a>
## 5. Полиномиальные и интерактивные признаки

**Обоснование:** Взаимодействия между признаками могут выявить нелинейные зависимости. Например, старые автомобили с большим пробегом могут терять стоимость быстрее.

In [ ]:
X_te['car_age_squared'] = X_te['car_age'] ** 2
X_te['log_odometer_squared'] = X_te['log_odometer'] ** 2

X_te['age_x_odometer'] = X_te['car_age'] * X_te['log_odometer']
X_te['age_x_cylinders'] = X_te['car_age'] * X_te['cylinders_num']
X_te['odometer_x_cylinders'] = X_te['log_odometer'] * X_te['cylinders_num']

X_te['mileage_per_year'] = X_te['odometer'] / (X_te['car_age'] + 1)  # +1 чтобы избежать деления на 0
X_te['log_mileage_per_year'] = np.log1p(X_te['mileage_per_year'])

print(f"Созданы полиномиальные и интерактивные признаки:")
print("   - car_age_squared")
print("   - log_odometer_squared")
print("   - age_x_odometer")
print("   - age_x_cylinders")
print("   - odometer_x_cylinders")
print("   - mileage_per_year")
print("   - log_mileage_per_year")

<a id='section-6'></a>
## 6. Скейлинг числовых признаков

**Обоснование:** Многие алгоритмы машинного обучения (особенно основанные на градиентах и расстояниях) чувствительны к масштабу признаков. StandardScaler приводит признаки к нулевому среднему и единичной дисперсии.

In [ ]:
NUMERICAL_COLS_TO_SCALE = [
    'car_age', 'log_odometer', 'cylinders_num',
    'car_age_squared', 'log_odometer_squared',
    'age_x_odometer', 'age_x_cylinders', 'odometer_x_cylinders',
    'mileage_per_year', 'log_mileage_per_year',
    'region_mean_price', 'region_median_price', 'region_price_std',
    'manuf_mean_price', 'manuf_median_price', 'manuf_price_std',
    'dist_to_region_center',
    'manufacturer_target', 'model_target', 'region_target'
]

cols_to_scale = [col for col in NUMERICAL_COLS_TO_SCALE if col in X_te.columns]

print(f"Числовые признаки для скейлинга: {len(cols_to_scale)}")
print(cols_to_scale)

In [ ]:
scaler = StandardScaler()
X_te.loc[:, cols_to_scale] = scaler.fit_transform(X_te[cols_to_scale])

print(f"Скейлинг применен")
print(f"Пример до и после (car_age):")
print(f"Среднее: {X_te['car_age'].mean():.4f}")
print(f"Стд: {X_te['car_age'].std():.4f}")

<a id='section-7'></a>
## 7. Отбор признаков

**Обоснование:** Не все признаки одинаково полезны. Некоторые могут быть шумом или сильно коррелировать друг с другом. Отбор признаков помогает:
- Уменьшить переобучение
- Ускорить обучение моделей
- Улучшить интерпретируемость

In [ ]:
COLS_TO_DROP = [
    'year', 'odometer', 'lat', 'long', 'region', 'manufacturer', 
    'model', 'cylinders', 'manufacturer_grouped', 'model_grouped',
    'lat_region_mean', 'long_region_mean', 'mileage_per_year'
]

X_final = X_te.drop(columns=[col for col in COLS_TO_DROP if col in X_te.columns])

print(f"Размерность после удаления исходных колонок: {X_final.shape}")

In [ ]:
X_temp = X_final.copy()
X_temp['price'] = y_log

correlations = X_temp.corr()['price'].drop('price').abs().sort_values(ascending=False)

print(f"Топ-20 признаков по корреляции с log(price):")
print(correlations.head(20))

In [ ]:
X_mi = X_final.fillna(0)

mi_scores = mutual_info_regression(X_mi, y_log, random_state=42)
mi_df = pd.DataFrame({
    'feature': X_final.columns,
    'mutual_info': mi_scores
}).sort_values('mutual_info', ascending=False)

print(f"Топ-20 признаков по Mutual Information:")
print(mi_df.head(20))

In [ ]:
selector = SelectKBest(score_func=f_regression, k=50)
selector.fit(X_mi, y_log)

selected_mask = selector.get_support()
selected_features = X_final.columns[selected_mask].tolist()

print(f"Выбрано {len(selected_features)} признаков из {X_final.shape[1]}")
print(f"Выбранные признаки:")
print(selected_features)

In [ ]:
X_best = X_final[selected_features].copy()

print(f"Финальная размерность: {X_best.shape}")
print(f"Целевая переменная: {y_log.shape}")

<a id='section-8'></a>
## 8. Сохранение результатов

In [ ]:
output_path = Path("../data/processed/train_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

X_best.to_csv(output_path)

y_log.to_csv(Path("../data/processed/train_target.csv"))

import joblib
joblib.dump(scaler, Path("../models/scaler.pkl"))

print(f"Данные сохранены в {output_path}")

In [ ]:
print("\n" + "="*70)
print("FEATURE ENGINEERING ЗАВЕРШЕН!")
print("="*70)
print(f"\nИсходные данные: {df.shape}")
print(f"Финальные признаки: {X_best.shape}")
print(f"Целевая переменная: {y_log.shape}")
print(f"\nСозданные признаки:")
print(f"   1. car_age — возраст автомобиля")
print(f"   2. log_odometer — логарифм пробега")
print(f"   3. cylinders_num — числовое представление цилиндров")
print(f"   4. One-Hot Encoded признаки для низкокардинальных категориальных переменных")
print(f"   5. Target Encoded признаки для высококардинальных переменных")
print(f"   6. Региональные статистики (mean, median, std, cluster)")
print(f"   7. Статистики по производителю")
print(f"   8. Полиномиальные признаки (age^2, odometer^2)")
print(f"   9. Интерактивные признаки (age x odometer, etc.)")
print(f"   10. Скейленные числовые признаки")
print(f"\nОтбор признаков:")
print(f"   Использованы: корреляционный анализ, Mutual Information, SelectKBest")
print(f"   Итоговое количество признаков: {X_best.shape[1]}")
print("\nГотово к обучению моделей!")